# 90 — Approval Gate Pattern

**Pattern:** Interrupt-and-resume human approval gate, built on LangGraph's real `StateGraph` + `interrupt()` + `Command(resume=...)`.  
**Key insight:** "Route to an approval tier" is a label on an output field. An actual *gate* means the tool-call boundary is unreachable until a human decision comes back in — that's a different, stronger claim.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/90-approval-gate-pattern/approval_gate_workbook.ipynb)

In [ ]:
%pip install -q langgraph langchain-openai langchain-core pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 1. Why a routing label is not a gate

Plenty of agent demos add a field like `approval_tier: "auto" | "manager" | "controller"` to their output schema and call that "human in the loop." It isn't — nothing stops the next line of code from firing the tool regardless of what that field says.

A real gate needs three things:
1. Execution **actually pauses** — not just "the agent suggests pausing"
2. The pause **survives** until a human responds (it can't just be a `time.sleep()`)
3. The human's decision (approve / edit / reject) **determines what executes next**, including the option to execute something *other* than what was originally proposed

LangGraph's `interrupt()` + a checkpointer gives you exactly this, for free.

## 2. The schema — ProposedAction, ApprovalDecision, ActionResult

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field


class ProposedAction(BaseModel):
    action_type: Literal["post_journal_entry", "revoke_access", "send_collection_notice"] = Field(
        description="The category of irreversible action being proposed"
    )
    summary: str = Field(description="One-sentence human-readable summary of what this action does")
    payload: dict = Field(description="The action-specific parameters a downstream system would execute")
    risk_level: Literal["low", "medium", "high"] = Field(description="Financial/operational blast radius")


class ApprovalDecision(BaseModel):
    decision: Literal["approve", "edit", "reject"] = Field(description="The human reviewer's resolution")
    edited_payload: Optional[dict] = Field(default=None, description="Replacement payload when decision='edit'")
    rationale: str = Field(description="The human's reason -- always logged, even on approve")


class ActionResult(BaseModel):
    executed: bool = Field(description="True only if the action actually fired")
    final_payload: Optional[dict] = Field(default=None, description="What was actually executed, after any edit")
    decision_log: str = Field(description="Audit-trail entry: what was proposed, decided, and why")

## 3. The graph — draft_action → human_review → execute_or_log

`human_review` is the whole lesson: calling `interrupt(payload)` halts the graph mid-run. The `payload` you pass becomes visible to whatever called `.invoke()` via `result["__interrupt__"][0].value`. Nothing downstream of `interrupt()` executes until a *separate* `.invoke(Command(resume=...))` call comes in on the same `thread_id`.

In [ ]:
import uuid
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

_MODEL = "gpt-4.1-nano"

DRAFT_SYSTEM = (
    "You turn a free-text business event into a single proposed irreversible action. "
    "Choose action_type: post_journal_entry, revoke_access, or send_collection_notice. "
    "Populate payload with concrete parameters. Set risk_level by blast radius."
)


class GraphState(TypedDict):
    event_description: str
    proposed_action: dict
    decision: dict
    result: dict


def draft_action(state: GraphState) -> dict:
    llm = ChatOpenAI(model=_MODEL, temperature=0)
    proposed = llm.with_structured_output(ProposedAction, method="function_calling").invoke(
        [("system", DRAFT_SYSTEM), ("human", state["event_description"])]
    )
    return {"proposed_action": proposed.model_dump()}


def human_review(state: GraphState) -> dict:
    decision = interrupt({"proposed_action": state["proposed_action"]})
    return {"decision": decision}


def execute_or_log(state: GraphState) -> dict:
    proposed = state["proposed_action"]
    decision = ApprovalDecision.model_validate(state["decision"])
    if decision.decision == "reject":
        result = ActionResult(executed=False, final_payload=None,
            decision_log=f"REJECTED: {proposed['summary']!r}. Reason: {decision.rationale}")
    else:
        final_payload = decision.edited_payload if decision.decision == "edit" and decision.edited_payload else proposed["payload"]
        verb = "EDITED-THEN-EXECUTED" if decision.decision == "edit" else "EXECUTED"
        result = ActionResult(executed=True, final_payload=final_payload,
            decision_log=f"{verb}: {proposed['summary']!r}. Reason: {decision.rationale}")
    return {"result": result.model_dump()}


graph = StateGraph(GraphState)
graph.add_node("draft_action", draft_action)
graph.add_node("human_review", human_review)
graph.add_node("execute_or_log", execute_or_log)
graph.add_edge(START, "draft_action")
graph.add_edge("draft_action", "human_review")
graph.add_edge("human_review", "execute_or_log")
graph.add_edge("execute_or_log", END)
app = graph.compile(checkpointer=InMemorySaver())


def propose(event_description: str):
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"event_description": event_description}, config=config)
    proposed = ProposedAction.model_validate(result["__interrupt__"][0].value["proposed_action"])
    return proposed, thread_id


def resume(thread_id: str, decision: ApprovalDecision):
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(Command(resume=decision.model_dump()), config=config)
    return ActionResult.model_validate(result["result"])

## 4. Run it — propose() pauses, resume() finishes

Two scenarios: one clean approve, one reject. Notice the proposed action prints *before* any decision exists — that's the pause being real, not simulated.

In [ ]:
proposed, tid = propose("Post a $4,200 accrual for Q2 consulting fees to cost centre CC1001.")
print("PAUSED — proposed action:")
print(proposed)
print()

decision = ApprovalDecision(decision="approve", rationale="Matches the signed SOW.")
result = resume(tid, decision)
print("RESUMED — result:")
print(result)

In [ ]:
proposed2, tid2 = propose("Send a final notice collection letter to ACME Corp for invoice INV-9981, $58,000 outstanding 95 days.")
print("PAUSED — proposed action:")
print(proposed2)
print()

decision2 = ApprovalDecision(decision="reject", rationale="Customer has an active payment plan on file -- escalate to account manager first.")
result2 = resume(tid2, decision2)
print("RESUMED — result:")
print(result2)

## 5. Starter Exercise

Nothing in `resume()` currently stops a human from rejecting an action with an empty rationale — a silent reject. Write a `safe_resume(thread_id, decision)` wrapper that raises `ValueError` if `decision.decision == "reject"` and `decision.rationale` is blank, otherwise calls `resume()` normally. Test it with an empty-rationale reject and confirm it raises.

In [ ]:
# Your code here

### Answer Key

In [ ]:
def safe_resume(thread_id: str, decision: ApprovalDecision):
    if decision.decision == "reject" and not decision.rationale.strip():
        raise ValueError("Rejection requires a non-empty rationale -- no silent rejects.")
    return resume(thread_id, decision)


proposed3, tid3 = propose("Revoke admin access for contractor CTR-9001, contract ended yesterday.")
try:
    safe_resume(tid3, ApprovalDecision(decision="reject", rationale=""))
except ValueError as e:
    print(f"Blocked as expected: {e}")